In [ ]:
# Cell 1: Imports 
import os, glob, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, precision_score, recall_score, accuracy_score

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

from PIL import Image

# Reproducibility
SEED = 42
np.random.seed(SEED)
random.seed(SEED)
tf.random.set_seed(SEED)


In [ ]:
# Cell 2: Image Loading (normalized 0–1)

def load_images_with_paths(folder, label, target_size=(128, 128)):
    images, labels, image_paths = [], [], []
    exts = (".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff")
    for file_name in sorted(os.listdir(folder)):
        path = os.path.join(folder, file_name)
        if path.lower().endswith(exts):
            img = load_img(path, target_size=target_size, color_mode="rgb")
            arr = img_to_array(img).astype("float32") / 255.0  # 🔧 normalization
            images.append(arr)
            labels.append(label)
            image_paths.append(path)
    return images, labels, image_paths

folder_good = "../data/good"
folder_bad  = "../data/bad"

good_imgs, y_good, paths_good = load_images_with_paths(folder_good, label=1)
bad_imgs,  y_bad,  paths_bad  = load_images_with_paths(folder_bad, label=0)

X = np.array(good_imgs + bad_imgs, dtype="float32")
y = np.array(y_good + y_bad, dtype="int32")
all_paths = np.array(paths_good + paths_bad)

print(f"Total loaded images: {len(X)}")
print(f"Image shape: {X[0].shape}")
unique_classes, counts = np.unique(y, return_counts=True)
print(f"Class Distribution (label:count): {dict(zip(unique_classes, counts))}")

In [ ]:
# Cell 3: Split (train/val/test) + sanity checks

X_train_val, X_test, y_train_val, y_test, paths_train_val, paths_test = train_test_split(
    X, y, all_paths, test_size=0.30, stratify=y, random_state=SEED
)

X_train, X_val, y_train, y_val, paths_train, paths_val = train_test_split(
    X_train_val, y_train_val, paths_train_val,
    test_size=0.132, stratify=y_train_val, random_state=SEED
)

def dataset_summary(name, y_data):
    unique_cls, counts = np.unique(y_data, return_counts=True)
    print(f"\n🔹 {name.upper()} - Total: {len(y_data)} images")
    for c, count in zip(unique_cls, counts):
        label_name = "CONFORMING" if c == 1 else "defect"
        print(f"  - {label_name}: {count} images")

dataset_summary("Train", y_train)
dataset_summary("Validation", y_val)
dataset_summary("Test", y_test)

# Leakage and scale checks
overlap_val  = set(paths_train) & set(paths_val)
overlap_test = set(paths_train) & set(paths_test)
print("\nIntersection train–Validation:", len(overlap_val))
print("Intersection train–Test    :", len(overlap_test))
if overlap_val:
    print("⚠️ Leakage train–Validation (examples):", list(sorted(overlap_val))[:5])
if overlap_test:
    print("⚠️ Leakage train–Test (examples):", list(sorted(overlap_test))[:5])

print("\nScale X_train (min,max):", float(X_train.min()), float(X_train.max()))
print("Scale X_val   (min,max):", float(X_val.min()),   float(X_val.max()))
print("Scale X_test  (min,max):", float(X_test.min()),  float(X_test.max()))

# Class weights (if imbalanced)
from collections import Counter
counter_dict = Counter(y_train.tolist())
maj = max(counter_dict.values()); minc = min(counter_dict.values())
if minc > 0 and (maj / minc) >= 1.5:
    cw = {0: maj / counter_dict.get(0, 1), 1: maj / counter_dict.get(1, 1)}
    print("\nclass_weight applied:", cw)
else:
    cw = None
    print("\nclass_weight unnecessary (balanced).")


In [ ]:
# Cell 4: CNN Architecture

model = Sequential([
    Conv2D(32, (3, 3), activation='relu', input_shape=X_train.shape[1:]),
    MaxPooling2D(2, 2),

    Conv2D(64, (3, 3), activation='relu'),
    MaxPooling2D(2, 2),

    Conv2D(128, (3, 3), activation='relu'),
    MaxPooling2D(2, 2),

    Flatten(),
    Dense(256, activation='relu'),
    Dropout(0.5),
    Dense(1, activation='sigmoid')
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.summary()


In [ ]:
# Cell 5: Training (correct validation_data + callbacks)

callbacks = [
    EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True),
    ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=5, min_lr=1e-6),
]

fit_kwargs = dict(
    x=X_train, y=y_train,
    validation_data=(X_val, y_val),   # 🔒 Validation set isolation
    epochs=100,
    batch_size=32,
    shuffle=True,
    callbacks=callbacks,
    verbose=1
)

if cw is not None:
    fit_kwargs["class_weight"] = cw

history = model.fit(**fit_kwargs)

# Sanity check: initial losses shouldn't be zero
print("loss[:3]    =", [round(v, 6) for v in history.history["loss"][:3]])
print("val_loss[:3]=", [round(v, 6) for v in history.history["val_loss"][:3]])


In [ ]:
# Cell — Plot Accuracy and Loss Curves (with moving averages)

import numpy as np
import matplotlib.pyplot as plt

def _moving_avg(x, k=9):
    x = np.asarray(x, dtype=float)
    if x.size == 0 or k <= 1: 
        return x
    w = np.ones(k, dtype=float) / float(k)
    return np.convolve(x, w, mode="same")

def plot_history_en(history, window_size=9):
    # Compatibility across Keras versions
    tr_acc = history.history.get("accuracy", history.history.get("acc", []))
    va_acc = history.history.get("val_accuracy", history.history.get("val_acc", []))
    tr_loss = history.history.get("loss", [])
    va_loss = history.history.get("val_loss", [])

    if not (len(tr_acc) and len(va_acc) and len(tr_loss) and len(va_loss)):
        print("⚠️ Missing history keys. Check if training successfully concluded.")
        print("Available Keys:", list(history.history.keys()))
        return

    # Moving averages (dashed)
    tr_acc_ma = _moving_avg(tr_acc, window_size)
    va_acc_ma = _moving_avg(va_acc, window_size)
    tr_loss_ma = _moving_avg(tr_loss, window_size)
    va_loss_ma = _moving_avg(va_loss, window_size)

    plt.figure(figsize=(12, 4))

    # --- Accuracy Plot ---
    ax1 = plt.subplot(1, 2, 1)
    ax1.plot(tr_acc, label="Train", alpha=0.8)
    ax1.plot(va_acc, label="Validation", alpha=0.8)
    ax1.plot(tr_acc_ma, "--", label="Train Moving Avg")
    ax1.plot(va_acc_ma, "--", label="Val Moving Avg")
    ax1.set_title("Accuracy Evolution")
    ax1.set_xlabel("Epochs")
    ax1.set_ylabel("Accuracy")
    ax1.grid(True, alpha=0.3)
    ax1.legend()

    # --- Loss Plot ---
    ax2 = plt.subplot(1, 2, 2)
    ax2.plot(tr_loss, label="Train", alpha=0.8)
    ax2.plot(va_loss, label="Validation", alpha=0.8)
    ax2.plot(tr_loss_ma, "--", label="Train Moving Avg")
    ax2.plot(va_loss_ma, "--", label="Val Moving Avg")
    ax2.set_title("Loss Evolution")
    ax2.set_xlabel("Epochs")
    ax2.set_ylabel("Loss")
    ax2.grid(True, alpha=0.3)
    ax2.legend()

    plt.tight_layout()
    plt.show()

# Call directly post-fit:
# plot_history_en(history, window_size=3)


In [ ]:
# Cell 6: Final Evaluation and Confusion Matrix

# Output Predictions
y_pred_probs = model.predict(X_test)
y_pred = (y_pred_probs > 0.5).astype("int32").flatten()

labels = ["defect", "good"]

cm = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()
plt.figure(figsize=(6, 6))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels)
disp.plot(cmap='Blues')
plt.title("Confusion Matrix")
plt.tight_layout()
plt.show()

# Manual Metric Calculation
acc = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average=None)
recall = recall_score(y_test, y_pred, average=None)

# Formatted Display
print(f"{'Class':<10} {'Precision':>10} {'Recall':>10}")
for idx, class_name in enumerate(labels):
    print(f"{class_name:<10} {precision[idx]:10.2f} {recall[idx]:10.2f}")

print(f"\nOverall Accuracy:  {acc:.2f}")